 # Homework 7: Quantum Teleportation & Superdense Coding



 **Name:** Marc Abi Zeid Daou



 **Date:** 4/6/2026

In [1]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram, plot_bloch_multivector
from qiskit.quantum_info import Statevector, DensityMatrix, partial_trace, state_fidelity
import numpy as np
import matplotlib.pyplot as plt

simulator = AerSimulator()

def run_circuit(qc, shots=1024):
    """Run a circuit and return measurement counts."""
    job = simulator.run(transpile(qc, simulator), shots=shots)
    return job.result().get_counts()


 ---

 ## Problem 1: Teleportation with Different Bell States (25 points)



 In lecture, we used |Φ⁺⟩ = (|00⟩ + |11⟩)/√2 as the entangled resource.

 But teleportation works with ANY of the four Bell states — with modified corrections.

In [15]:
# 
# Implement teleportation using |Ψ⁺⟩ = (|01⟩ + |10⟩)/√2 instead of |Φ⁺⟩.
# You'll need to figure out the correct correction operations.

def teleport_with_psi_plus(state_prep_gates=None):
    """
    Teleportation using |Ψ⁺⟩ = (|01⟩ + |10⟩)/√2 as the entangled resource.
    
    Args:
        state_prep_gates: List of (gate_name, params) tuples to prepare state on q0
                         Example: [('h', None), ('s', None)]
    
    Returns:
        QuantumCircuit implementing teleportation with |Ψ⁺⟩
    
    Hint: |Ψ⁺⟩ can be created from |Φ⁺⟩ by applying X to one qubit.
    The correction operations will be DIFFERENT from the |Φ⁺⟩ case.
    Work out the math to determine the new corrections!
    """
    qr = QuantumRegister(3, 'q')
    crz = ClassicalRegister(1, 'crz')
    crx = ClassicalRegister(1, 'crx')
    qc = QuantumCircuit(qr, crz, crx)
    
    # TODO: Create |Ψ⁺⟩ Bell pair between q1 and q2
    # Your code here
    qc.h(1)           # q1 in |+> state
    qc.cx(1, 2)       # CNOT creates (|00> + |11>)/sqrt(2)
    qc.x(1)           # X gate transforms to (|01> + |10>)/sqrt(2) = |Ψ⁺⟩
    
    qc.barrier()
    
    # TODO: Prepare the state to teleport on q0
    if state_prep_gates:
        for gate, params in state_prep_gates:
            if gate == 'h': qc.h(0)
            elif gate == 'x': qc.x(0)
            elif gate == 'y': qc.y(0)
            elif gate == 'z': qc.z(0)
            elif gate == 's': qc.s(0)
            elif gate == 't': qc.t(0)
            elif gate == 'rx': qc.rx(params, 0)
            elif gate == 'ry': qc.ry(params, 0)
            elif gate == 'rz': qc.rz(params, 0)
    
    qc.barrier()
    
    # TODO: Bell measurement on q0 and q1 (same as standard teleportation)
    # Your code here
    qc.cx(0, 1)
    qc.h(0)
    qc.measure([0, 1], [crx[0], crz[0]])
    
    qc.barrier()
    
    # TODO: Apply CORRECTED corrections for |Ψ⁺⟩ case
    # These will be different from the |Φ⁺⟩ corrections!
    # Work out the math: what state does Bob have for each measurement outcome?
    # Your code here
    with qc.if_test((crx, 1)):
        qc.z(2)
    with qc.if_test((crz, 1)):
        qc.x(2)
    qc.x(2) 
    return qc

# Test your implementation
print("Testing |Ψ⁺⟩ teleportation:")
qc_test = teleport_with_psi_plus([('h', None)])
print(qc_test.draw())


Testing |Ψ⁺⟩ teleportation:
                       ░ ┌───┐ ░      ┌───┐┌─┐ ░                        »
  q_0: ────────────────░─┤ H ├─░───■──┤ H ├┤M├─░────────────────────────»
       ┌───┐     ┌───┐ ░ └───┘ ░ ┌─┴─┐└┬─┬┘└╥┘ ░                        »
  q_1: ┤ H ├──■──┤ X ├─░───────░─┤ X ├─┤M├──╫──░────────────────────────»
       └───┘┌─┴─┐└───┘ ░       ░ └───┘ └╥┘  ║  ░ ┌────── ┌───┐ ───────┐ »
  q_2: ─────┤ X ├──────░───────░────────╫───╫──░─┤ If-0  ┤ Z ├  End-0 ├─»
            └───┘      ░       ░        ║   ║  ░ └──╥─── └───┘ ───────┘ »
crz: 1/═════════════════════════════════╩═══╬═══════╬═══════════════════»
                                        0   ║    ┌──╨──┐                »
crx: 1/═════════════════════════════════════╩════╡ 0x1 ╞════════════════»
                                            0    └─────┘                »
«                                   
«  q_0: ────────────────────────────
«                                   
«  q_1: ────────────────────────────
«       ┌─

In [14]:
# 
# Verify your |Ψ⁺⟩ teleportation works by comparing results with expected values.

def verify_psi_plus_teleportation():
    """
    Test teleportation with |Ψ⁺⟩ for multiple states.
    Returns dictionary of results.
    """
    test_cases = [
        (None, "|0⟩"),
        ([('x', None)], "|1⟩"),
        ([('h', None)], "|+⟩"),
        ([('h', None), ('s', None)], "|i⟩"),
    ]
    
    results = {}
    
    for prep, name in test_cases:
        # TODO: Create circuit with measurement of Bob's qubit
        qc = teleport_with_psi_plus(prep)
        
        # TODO: Add output measurement register and measure q2
        # Your code here
        
        # TODO: Run and extract output qubit statistics
        # Your code here
        cr_out = ClassicalRegister(1, 'out')
        qc.add_register(cr_out)
        qc.measure(2, cr_out)
        counts = run_circuit(qc, shots=1000)
        results[name] = counts
        
    return results

# Run verification
print("\n|Ψ⁺⟩ Teleportation Results:")
print("-" * 40)
results = verify_psi_plus_teleportation()
for state, counts in results.items():
    print(f"{state}: {counts}")



|Ψ⁺⟩ Teleportation Results:
----------------------------------------
|0⟩: {'1 0 1': 276, '1 1 1': 244, '1 1 0': 237, '1 0 0': 243}
|1⟩: {'0 1 1': 262, '0 0 0': 264, '0 1 0': 213, '0 0 1': 261}
|+⟩: {'0 0 0': 136, '1 0 1': 120, '1 1 1': 126, '0 1 1': 140, '0 1 0': 111, '1 0 0': 109, '1 1 0': 115, '0 0 1': 143}
|i⟩: {'1 0 1': 134, '0 1 1': 120, '0 0 1': 118, '0 1 0': 127, '1 1 0': 118, '1 1 1': 130, '0 0 0': 130, '1 0 0': 123}


In [16]:
import sys
print(sys.executable)

C:\Users\marcd\qiskit_env\Scripts\python.exe


 **Question 1C:** Show the mathematical derivation for why your corrections

 work with |Ψ⁺⟩. Specifically:



 1. Write out the initial 3-qubit state |ψ⟩ ⊗ |Ψ⁺⟩

 2. Apply CNOT and H (Bell measurement gates)

 3. Regroup by Alice's measurement outcomes

 4. Show what corrections Bob needs for each outcome



 *Your derivation here:*
|Ψ⁺⟩ = (|01⟩ + |10⟩)/√2

 Initial: |ψ⟩ ⊗ |Ψ⁺⟩ where |ψ⟩ = α|0⟩ + β|1⟩

 After Bell measurement (CNOT + H on q0,q1):
 - Measurement 00 → q2 = |ψ⟩ ✓
 - Measurement 01 → q2 = X|ψ⟩ (apply X to fix)
 - Measurement 10 → q2 = Z|ψ⟩ (apply Z to fix)
 - Measurement 11 → q2 = ZX|ψ⟩ (apply both to fix)




 ---

 ## Problem 2: Superdense Coding Variations (25 points)

In [ ]:
#
# In lecture, we built encoding AND decoding together. Here, implement
# ONLY the decoder — a function that takes an already-encoded Bell state
# and extracts the 2-bit message.

def superdense_decoder(input_state):
    """
    Decode a superdense-coded message from a Bell state.
    
    Args:
        input_state: String identifying the Bell state
                    'phi_plus', 'phi_minus', 'psi_plus', 'psi_minus'
    
    Returns:
        Decoded 2-bit message as string ('00', '01', '10', or '11')
    
    This simulates Bob receiving an already-encoded state and decoding it.
    """
    qc = QuantumCircuit(2, 2)
    
    # Create the input Bell state (simulating what Bob receives)
    # This represents the state AFTER Alice's encoding
    if input_state == 'phi_plus':
        qc.h(0)
        qc.cx(0, 1)
    elif input_state == 'phi_minus':
        qc.h(0)
        qc.cx(0, 1)
        qc.z(0)
    elif input_state == 'psi_plus':
        qc.h(0)
        qc.cx(0, 1)
        qc.x(0)
    elif input_state == 'psi_minus':
        qc.h(0)
        qc.cx(0, 1)
        qc.z(0)
        qc.x(0)
    
    qc.barrier()
    
    # TODO: Implement Bob's decoding (Bell measurement)
    # Your code here
    
    # TODO: Run circuit and return the decoded message
    # Your code here
    
    return "00"  # Placeholder - replace with actual result

# Test all four cases
print("Superdense Decoder Test:")
print("-" * 40)
expected = {
    'phi_plus': '00',
    'phi_minus': '10', 
    'psi_plus': '01',
    'psi_minus': '11'
}
for state, exp in expected.items():
    result = superdense_decoder(state)
    status = "✓" if result == exp else "✗"
    print(f"{state} → {result} (expected {exp}) {status}")


In [ ]:
#
# What happens if Bob uses the WRONG decoding strategy?

def analyze_wrong_decoding():
    """
    Analyze what happens when Bob decodes incorrectly.
    
    Instead of CNOT then H, what if Bob does:
    1. Only H (no CNOT)
    2. Only CNOT (no H)
    3. H then CNOT (reversed order)
    
    For each wrong method, determine the mapping from sent → received.
    """
    results = {
        'only_h': {},
        'only_cnot': {},
        'reversed_order': {}
    }
    
    messages = ['00', '01', '10', '11']
    
    for msg in messages:
        # Create properly encoded state
        for method in results.keys():
            qc = QuantumCircuit(2, 2)
            
            # Encode (correct encoding)
            qc.h(0)
            qc.cx(0, 1)
            if msg[1] == '1': qc.x(0)
            if msg[0] == '1': qc.z(0)
            qc.barrier()
            
            # TODO: Apply WRONG decoding based on method
            if method == 'only_h':
                # Your code here
                pass
            elif method == 'only_cnot':
                # Your code here
                pass
            elif method == 'reversed_order':
                # Your code here
                pass
            
            qc.measure([0, 1], [0, 1])
            
            # TODO: Run and record most common result
            # Your code here
            
            results[method][msg] = "??"  # Replace with actual result
    
    return results

# Run analysis
print("\nWrong Decoding Analysis:")
print("=" * 50)
wrong_results = analyze_wrong_decoding()
for method, mapping in wrong_results.items():
    print(f"\n{method}:")
    for sent, received in mapping.items():
        status = "✓" if sent == received else "✗"
        print(f"  Sent {sent} → Received {received} {status}")


 **Question 2C-i:** For each wrong decoding method, is ANY information

 preserved? Could Bob recover the original message with post-processing?



 *Your answer here:*





 **Question 2C-ii:** The "reversed order" decoding (H then CNOT) is actually

 a valid quantum operation. What transformation does it perform? Why doesn't

 it correctly decode superdense coding?



 *Your answer here:*



 ---

 ## Problem 3: Protocol Debugging (20 points)



 Debug broken implementations of both protocols.

In [ ]:
#
# The following teleportation circuit has THREE bugs. Find and fix them.

def broken_teleportation():
    """
    This teleportation circuit has 3 bugs. Find and fix them all.
    
    Bugs could be in:
    - Bell pair creation
    - Bell measurement
    - Corrections
    - Qubit/bit ordering
    """
    qr = QuantumRegister(3, 'q')
    crz = ClassicalRegister(1, 'crz')
    crx = ClassicalRegister(1, 'crx')
    qc = QuantumCircuit(qr, crz, crx)
    
    # Create Bell pair (BUG #1 is here)
    qc.h(1)
    qc.cx(2, 1)  # Something wrong here...
    qc.barrier()
    
    # Prepare |+⟩ to teleport
    qc.h(0)
    qc.barrier()
    
    # Bell measurement (BUG #2 is here)
    qc.cx(1, 0)  # Something wrong here...
    qc.h(0)
    qc.measure(0, crz)
    qc.measure(1, crx)
    qc.barrier()
    
    # Corrections (BUG #3 is here)
    qc.x(2).c_if(crz, 1)  # Something wrong here...
    qc.z(2).c_if(crx, 1)
    
    return qc

def fixed_teleportation():
    """
    TODO: Implement the corrected version of the teleportation circuit.
    """
    qr = QuantumRegister(3, 'q')
    crz = ClassicalRegister(1, 'crz')
    crx = ClassicalRegister(1, 'crx')
    qc = QuantumCircuit(qr, crz, crx)
    
    # TODO: Fix Bug #1 - Bell pair creation
    # Your code here
    
    qc.barrier()
    
    # Prepare |+⟩ to teleport
    qc.h(0)
    qc.barrier()
    
    # TODO: Fix Bug #2 - Bell measurement
    # Your code here
    
    qc.barrier()
    
    # TODO: Fix Bug #3 - Corrections
    # Your code here
    
    return qc


 **Describe the 3 bugs you found:**



 1. Bug #1:



 2. Bug #2:



 3. Bug #3:



In [ ]:
print("Testing fixed teleportation:")
qc_fixed = fixed_teleportation()
cr_out = ClassicalRegister(1, 'out')
qc_fixed.add_register(cr_out)
qc_fixed.measure(2, cr_out)

counts = run_circuit(qc_fixed, shots=2000)
print(f"Results (teleporting |+⟩): {counts}")
print("Expected: ~50% '0' and ~50% '1' in output bit")


In [ ]:
#
# This superdense coding has TWO bugs.

def broken_superdense(bits):
    """
    This superdense coding circuit has 2 bugs.
    """
    qc = QuantumCircuit(2, 2)
    
    # Create Bell pair
    qc.h(0)
    qc.cx(0, 1)
    qc.barrier()
    
    # Alice encodes (BUG #1 is here)
    if bits[0] == '1':  # Something wrong with bit ordering...
        qc.x(0)
    if bits[1] == '1':
        qc.z(0)
    qc.barrier()
    
    # Bob decodes (BUG #2 is here)
    qc.h(0)      # Something wrong with gate order...
    qc.cx(0, 1)
    qc.measure([0, 1], [0, 1])
    
    return qc

def fixed_superdense(bits):
    """
    TODO: Implement the corrected superdense coding.
    """
    qc = QuantumCircuit(2, 2)
    
    # Create Bell pair
    qc.h(0)
    qc.cx(0, 1)
    qc.barrier()
    
    # TODO: Fix Bug #1 - Alice's encoding
    # Your code here
    
    qc.barrier()
    
    # TODO: Fix Bug #2 - Bob's decoding
    # Your code here
    
    return qc

# Verify
print("\nTesting fixed superdense coding:")
for bits in ['00', '01', '10', '11']:
    qc = fixed_superdense(bits)
    counts = run_circuit(qc, shots=100)
    received = max(counts, key=counts.get)
    status = "✓" if received == bits else "✗"
    print(f"Sent: {bits} → Received: {received} {status}")


 **Describe the 2 bugs you found:**



 1. Bug #1:



 2. Bug #2:



 ---

 ## Problem 4: Chained Teleportation (15 points)



 What if we need to teleport through an intermediary?

In [ ]:
#
# Implement teleportation from Alice → Charlie → Bob
# (Alice teleports to Charlie, Charlie teleports to Bob)

def two_hop_teleportation(state_prep_gates=None):
    """
    Implement chained teleportation: Alice → Charlie → Bob
    
    Qubits:
    - q0: Alice's state to teleport
    - q1: Alice's half of Bell pair with Charlie
    - q2: Charlie's half of Bell pair with Alice (receives first)
    - q3: Charlie's half of Bell pair with Bob  
    - q4: Bob's half of Bell pair with Charlie (final destination)
    
    Classical registers for measurements and corrections.
    
    Returns:
        QuantumCircuit implementing two-hop teleportation
    """
    qr = QuantumRegister(5, 'q')
    # Classical bits for Alice→Charlie teleportation
    cr_ac_z = ClassicalRegister(1, 'ac_z')
    cr_ac_x = ClassicalRegister(1, 'ac_x')
    # Classical bits for Charlie→Bob teleportation
    cr_cb_z = ClassicalRegister(1, 'cb_z')
    cr_cb_x = ClassicalRegister(1, 'cb_x')
    
    qc = QuantumCircuit(qr, cr_ac_z, cr_ac_x, cr_cb_z, cr_cb_x)
    
    # TODO: Create Bell pair between Alice (q1) and Charlie (q2)
    # Your code here
    
    # TODO: Create Bell pair between Charlie (q3) and Bob (q4)
    # Your code here
    
    qc.barrier()
    
    # TODO: Prepare state on q0
    if state_prep_gates:
        for gate, params in state_prep_gates:
            if gate == 'h': qc.h(0)
            elif gate == 'x': qc.x(0)
            elif gate == 's': qc.s(0)
            elif gate == 'ry': qc.ry(params, 0)
    
    qc.barrier()
    
    # TODO: First teleportation: Alice (q0) → Charlie (q2)
    # Bell measurement on q0, q1
    # Your code here
    
    # TODO: Charlie's corrections on q2
    # Your code here
    
    qc.barrier()
    
    # TODO: Second teleportation: Charlie (q2) → Bob (q4)
    # Bell measurement on q2, q3
    # Your code here
    
    # TODO: Bob's corrections on q4
    # Your code here
    
    return qc

# Test
print("Two-hop teleportation circuit (teleporting |+⟩):")
qc_2hop = two_hop_teleportation([('h', None)])
print(qc_2hop.draw())

# Verify it works
cr_out = ClassicalRegister(1, 'out')
qc_2hop.add_register(cr_out)
qc_2hop.measure(4, cr_out)
counts = run_circuit(qc_2hop, shots=2000)
print(f"\nResults: {counts}")
print("Expected for |+⟩: ~50/50 distribution in output")


 **Question 4B-i:** In two-hop teleportation, how many Bell pairs are consumed?

 How many classical bits are transmitted in total?



 *Your answer:*





 **Question 4B-ii:** Could Alice and Bob achieve the same result with direct

 teleportation (if they shared a Bell pair)? What's the advantage of using

 an intermediary like Charlie?



 *Your answer:*





 **Question 4B-iii:** In a quantum network, Charlie might be a "quantum repeater."

 Why might we need repeaters for long-distance quantum communication?



 *Your answer:*



 ---

 ## Bonus: Entanglement Swapping (10 points)



 Entanglement swapping creates entanglement between particles that never interacted!

In [ ]:

def entanglement_swapping():
    """
    Implement entanglement swapping protocol.
    
    Setup:
    - Alice and Charlie share Bell pair (q0, q1)
    - Charlie and Bob share Bell pair (q2, q3)
    - Charlie performs Bell measurement on his qubits (q1, q2)
    - Result: Alice and Bob become entangled (q0, q3)!
    
    Returns:
        QuantumCircuit implementing the protocol
    """
    qc = QuantumCircuit(4, 4)
    
    # TODO: Create Bell pair between Alice (q0) and Charlie (q1)
    # Your code here
    
    # TODO: Create Bell pair between Charlie (q2) and Bob (q3)
    # Your code here
    
    qc.barrier()
    
    # TODO: Charlie performs Bell measurement on q1 and q2
    # Your code here
    
    qc.barrier()
    
    # TODO: Measure Alice's and Bob's qubits to verify entanglement
    # They should be correlated even though they never interacted!
    # Your code here
    
    return qc

# Test
print("Entanglement Swapping:")
qc_swap = entanglement_swapping()
print(qc_swap.draw())

counts = run_circuit(qc_swap, shots=2000)
print(f"\nResults: {counts}")


 **Bonus Questions:**



 1. What correlations do you observe between Alice's (q0) and Bob's (q3) measurements?



 2. Alice and Bob's particles never interacted directly. How is this possible?



 3. How is entanglement swapping related to two-hop teleportation?



 *Your answers:*



 ---

 ## Submission Checklist



 Before submitting, verify:

 - [ ] All code cells run without errors

 - [ ] All TODO sections are completed

 - [ ] All written questions are answered

 - [ ] Bug descriptions are complete and accurate

 - [ ] Output is visible for all verification tests

 - [ ] Bonus section attempted (optional)



 **Submit via Gradescope by Tuesday 11:59 PM**